In [1]:
import torch.nn as nn
import json
import torch

In [44]:
with open("config/config_TBR_land.json", "r") as f:
    config = json.load(f)

In [46]:
config['hidden_size']

64

In [34]:
def create_layer(layer_config):
    layer_type = layer_config["type"]
    params = {k: v for k, v in layer_config.items() if k != "type"}
    
    # Dynamically instantiate the layer
    if hasattr(nn, layer_type):
        return getattr(nn, layer_type)(**params)
    else:
        raise ValueError(f"Layer type {layer_type} is not supported.")

In [39]:
layers = [create_layer(layer_config) for layer_config in config["topo_fc"]]

In [40]:
layers

[Linear(in_features=128, out_features=64, bias=True),
 ReLU(),
 Linear(in_features=64, out_features=10, bias=True),
 Softmax(dim=1)]

In [41]:
model = nn.Sequential(*layers)

In [42]:
model(t)

tensor([[0.1104, 0.0937, 0.1084, 0.1149, 0.1129, 0.0970, 0.0857, 0.1158, 0.0793,
         0.0820],
        [0.1063, 0.0937, 0.1026, 0.1169, 0.1232, 0.0927, 0.0801, 0.1057, 0.0925,
         0.0864],
        [0.1104, 0.1122, 0.0958, 0.0991, 0.1090, 0.0960, 0.0929, 0.0950, 0.0909,
         0.0987],
        [0.1113, 0.1002, 0.1001, 0.1007, 0.1188, 0.0947, 0.0822, 0.1160, 0.0845,
         0.0915],
        [0.1122, 0.0948, 0.1025, 0.1119, 0.1148, 0.0924, 0.0848, 0.1255, 0.0784,
         0.0828],
        [0.1076, 0.0968, 0.0959, 0.1117, 0.1163, 0.0981, 0.0957, 0.1065, 0.0865,
         0.0849],
        [0.1027, 0.1073, 0.0971, 0.0999, 0.1145, 0.0931, 0.0966, 0.1054, 0.0784,
         0.1051],
        [0.1270, 0.0924, 0.0989, 0.0988, 0.1212, 0.0854, 0.0904, 0.0982, 0.0973,
         0.0904],
        [0.1180, 0.1005, 0.0899, 0.0980, 0.1013, 0.1004, 0.0942, 0.1074, 0.0848,
         0.1054],
        [0.1288, 0.0955, 0.0911, 0.1144, 0.1064, 0.0868, 0.0805, 0.1090, 0.0990,
         0.0886]], grad_fn=<

In [24]:
t = torch.rand([10,128])

In [26]:
tmp = torch.arange(128)
tmp_2 = torch.arange(128)
tmp_3 = torch.arange(128)

In [29]:
t = torch.vstack([tmp,tmp_2,tmp_3])

In [30]:
t.shape

torch.Size([3, 128])

In [53]:
after_split = torch.split(t,int(t.shape[1]/2),dim=1)

In [59]:
after_split[0]#.shape

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
         36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
         54, 55, 56, 57, 58, 59, 60, 61, 62, 63],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
         36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
         54, 55, 56, 57, 58, 59, 60, 61, 62, 63],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
         36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
         54, 55, 56, 57, 58, 59, 60, 61, 62, 63]])

In [52]:
len(after_split)

2

In [60]:
t = torch.rand([64, 1, 1000])

In [61]:
t.shape

torch.Size([64, 1, 1000])

In [71]:
torch.split(t,int(t.shape[2]/2),dim=2)[1].shape

torch.Size([64, 1, 500])

In [66]:
int(t.shape[1]/2)

0

In [85]:
from torchsummary import summary

In [92]:
def build_transformation_model():
    model = nn.Sequential()

    # Downsampling with Strided Convolution
    model.append(nn.Conv2d(in_channels = 1 ,out_channels = 16, kernel_size = (3, 3), stride=2,padding=1))
    model.append(nn.ReLU())
    model.append(nn.Conv2d(in_channels = 16 ,out_channels = 32, kernel_size = (3, 3), stride=1,padding=1))
    model.append(nn.ReLU())
    # Feature Expansion to 3 channels
    model.append(nn.Conv2d(in_channels = 32 ,out_channels = 3, kernel_size = (3, 3), stride=1,padding=1))  # Sigmoid for pixel intensity scaling

    return model

model = build_transformation_model()
model.to('cuda:0')
summary(model, (1,64,64))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 32, 32]             160
              ReLU-2           [-1, 16, 32, 32]               0
            Conv2d-3           [-1, 32, 32, 32]           4,640
              ReLU-4           [-1, 32, 32, 32]               0
            Conv2d-5            [-1, 3, 32, 32]             867
Total params: 5,667
Trainable params: 5,667
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.02
Forward/backward pass size (MB): 0.77
Params size (MB): 0.02
Estimated Total Size (MB): 0.81
----------------------------------------------------------------


In [91]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ImageTransformer(nn.Module):
    def __init__(self):
        super(ImageTransformer, self).__init__()
        
        # Downsampling: Reduce spatial dimensions from 64x64 to 32x32
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)  # (64x64x1 → 32x32x16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # (32x32x16 → 32x32x32)
        
        # Expanding the feature channels from 1 to 3
        self.conv3 = nn.Conv2d(32, 3, kernel_size=3, padding=1)  # (32x32x32 → 32x32x3)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.sigmoid(self.conv3(x))  # Sigmoid activation to ensure pixel values are between 0 and 1
        return x

# Instantiate model
model = ImageTransformer()

# Check model architecture
print(model)

# Example input tensor (batch size 1, channels 1, height 64, width 64)
x = torch.randn(1, 1, 64, 64)

# Forward pass
output = model(x)

# Output shape should be (1, 3, 32, 32)
print("Output shape:", output.shape)

ImageTransformer(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)
Output shape: torch.Size([1, 3, 32, 32])
